## Imports and Config

In [ ]:
%load_ext autoreload
%autoreload 2

Custom imports from the `config/settings.py` file.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

In [ ]:
import os
import json
import configparser
import pandas as pd
import torch

from haystack import Document
from haystack.document_stores.types import DuplicatePolicy
from haystack_integrations.document_stores.qdrant import QdrantDocumentStore

from openai import OpenAI

In [ ]:
from sem_rag.config.settings import DATA_DIR, VLLM, Qdrant, SEMDataset, LAMEDataset
from sem_rag.dataset.metadata import parse_category, parse_value_with_unit, parse_pixel_size, parse_magnification
from sem_rag.image.encoding import encode_image
from sem_rag.image.descriptions import load_descriptions, get_crop_path, get_sem_description, get_lame_description

Build both Qdrant collections (DINO and QWEN) with `payload_fields_to_index` set up properly, so metadata filtering (e.g. "AP_WD between 3.0 and 3.4") is actually fast at scale instead of falling back to an unindexed scan.

Without `payload_fields_to_index`, Qdrant *can* still filter on these fields, but it has to scan payloads rather than use an index.

field_schema values come straight from Qdrant's own payload index types:
https://qdrant.tech/documentation/concepts/indexing/#payload-index
- "float"   -> numeric range filtering (>=, <=) -- use for all 12 settings
- "integer" -> numeric range filtering for whole numbers -- use for SEMImageID
- "keyword" -> exact-match / categorical filtering -- use for ID-like strings

In [ ]:
client = OpenAI(base_url=VLLM.API_URL,api_key=VLLM.API_KEY)

At the moment, we are doing it locally (inside data/qdrant).

In [ ]:
QDRANT_PATH = DATA_DIR / "qdrant"

In [ ]:
def load_field_dictionaries():
    numeric_df = pd.read_csv(SEMDataset.NUMERICAL_SETTINGS, sep=";")
    categorical_df = pd.read_csv(SEMDataset.CATEGORICAL_SETTINGS, sep=";")

    numeric_map = {
        row["canonical_field"]: [
            field.strip()
            for field in row["json_fields"].split(",")
        ] for _, row in numeric_df.iterrows()
    }

    categorical_map = {
        row["canonical_field"]: [
            field.strip()
            for field in row["json_fields"].split(",")
        ] for _, row in categorical_df[categorical_df["canonical_field"] != "category"].drop_duplicates(subset=["canonical_field"]).iterrows()
    }

    return numeric_df, categorical_df, numeric_map, categorical_map

In [ ]:
NUMERIC_FIELDS_DF, CATEGORICAL_FIELDS_DF, NUMERIC_MAP, CATEGORICAL_MAP = load_field_dictionaries()

NUMERIC_FIELDS = NUMERIC_FIELDS_DF["canonical_field"].unique().tolist()
CATEGORICAL_FIELDS = CATEGORICAL_FIELDS_DF["canonical_field"].unique().tolist()

In [ ]:
categories_df = pd.read_csv(SEMDataset.CATEGORIES)
CATEGORY_MAP = dict(zip(categories_df["TIF_SEMImageID"], categories_df["mapped_category"]))

In [ ]:
NUMERIC_MAP

In [ ]:
CATEGORICAL_MAP

In [ ]:
def load_all_metadata(folder_path):
    """
    Take the JSON folder and parse each file to extract the ID and the `SETTINGS`.  
    Returns a Pandas dataframe where each row is a file and columns are its values.
    """
    all_rows = []

    for filename in sorted(os.listdir(folder_path)):
        if not filename.endswith(".json"):
            continue

        file_path = os.path.join(folder_path, filename)
        with open(file_path, "r") as f:
            data = json.load(f)

        image_id = int(data["SEMImageID"])

        row = {
            "SEMImageID": image_id,
            "ImageOriginalName": data["ImageOriginalName"],
            "ImageHashName": data["ImageHashName"],
            "ImageAVGHashName": data["ImageAVGHashName"],
        }

        # Numeric fields
        for canonical_field, json_fields in NUMERIC_MAP.items():
            value = None
            unit = None

            for json_field in json_fields:
                raw_value = data.get(json_field)

                if canonical_field == "pixel_size_nm":
                    parsed_value, parsed_unit = parse_pixel_size(raw_value, json_field)
                elif canonical_field == "magnification_x":
                    parsed_value, parsed_unit = parse_magnification(raw_value)
                else:
                    parsed_value, parsed_unit = parse_value_with_unit(raw_value)

                if parsed_value is not None:
                    value = parsed_value
                    unit = parsed_unit
                    break

            row[canonical_field] = value
            if unit:
                row[f"{canonical_field}_unit"] = unit

        # External category annotation
        row["category"] = CATEGORY_MAP.get(image_id)

        # Categorical field ("detector")
        for canonical_field, json_fields in CATEGORICAL_MAP.items():
            value = None

            for json_field in json_fields:
                raw_value = data.get(json_field)

                parsed_value = parse_category(raw_value)

                if parsed_value is not None:
                    value = parsed_value
                    break

            row[canonical_field] = value

        all_rows.append(row)

    df = pd.DataFrame(all_rows)

    print(f"Loaded {len(df)} files.")

    return df

In [ ]:
ordered_columns = [
    "SEMImageID",
    "ImageOriginalName",
    "ImageHashName",
    "ImageAVGHashName",
]

for canonical_field in NUMERIC_MAP:
    ordered_columns.append(canonical_field)

    unit_column = f"{canonical_field}_unit"
    ordered_columns.append(unit_column)

ordered_columns.extend([
    "category",
])

for canonical_field in CATEGORICAL_MAP:
    ordered_columns.append(canonical_field)

ordered_columns.remove("scan_speed_setting_unit")
ordered_columns.remove("frame_average_count_unit")
ordered_columns.remove("line_average_count_unit")

In [ ]:
df = load_all_metadata(SEMDataset.JSON)
df = df[ordered_columns]
df = df.sort_values(by='SEMImageID', ascending=True).reset_index(drop=True)
df.head()

In [ ]:
first_row = df.iloc[1000]
# (first_row.scan_speed_setting_unit)
first_row

In [ ]:
def load_embeddings(folder_path):
    """
    Take the embeddings (`.pt`) folder and load each file with `torch`.  
    Returns a dictionary of imageID: embedding.
    """
    embeddings = {}
    for filename in sorted(os.listdir(folder_path)):
        if not filename.endswith(".pt"):
            continue
        image_id = int(os.path.splitext(filename)[0])
        tensor = torch.load(os.path.join(folder_path, filename))
        embeddings[image_id] = tensor.flatten().tolist()
    return embeddings

In [ ]:
embeds = load_embeddings(SEMDataset.EMBEDDINGS_QWEN)

In [ ]:
dict(sorted(embeds.items(), key=lambda item: item[0]))

In [ ]:
def render_content(row):
    """
    Create textual content from canonical metadata.
    This content is provided to the LLM after retrieval.
    """
    parts = [f"SEM image {row['SEMImageID']}."]

    for field in NUMERIC_FIELDS:
        value = row.get(field)

        if pd.notna(value):
            unit = row.get(f"{field}_unit")

            if unit:
                parts.append(
                    f"{field}: {value} {unit}."
                )
            else:
                parts.append(
                    f"{field}: {value}."
                )

    for field in CATEGORICAL_FIELDS:
        value = row.get(field)

        if pd.notna(value):
            parts.append(
                f"{field}: {value}."
            )

    return " ".join(parts)

In [ ]:
render_content(df.iloc[1000])

In [ ]:
NUMERIC_FIELDS

In [ ]:
def build_meta(row, include_thumbnail=False):
    """
    Build the metadata for filtering to append to the Document to upload to Qdrant.  
    Returns a dictionary of key: value.
    """
    meta = {
        "SEMImageID": int(row["SEMImageID"]),
        "ImageOriginalName": row["ImageOriginalName"],
        "ImageHashName": row["ImageHashName"],
    }

    # Numeric metadata
    for field in NUMERIC_FIELDS:
        value = row.get(field)

        if pd.notna(value):
            meta[field] = float(value)

    # Categorical metadata
    for field in CATEGORICAL_FIELDS:
        value = row.get(field)

        if pd.notna(value):
            meta[field] = str(value)

    if include_thumbnail:
        crop_path = get_crop_path(int(row["SEMImageID"]))
        meta["thumb_b64"] = encode_image(crop_path, size=SEMDataset.THUMBNAIL_SIZE)

    return meta

In [ ]:
def build_documents(embeddings, json_folder, include_thumbnail=False):
    """
    Build the Haystack Document to upload to Qdrant.  
    Returns a list of Documents.
    """
    df = load_all_metadata(json_folder)
    descriptions = load_descriptions(SEMDataset.DESCRIPTIONS)
    documents = []
    skipped = []

    for _, row in df.iterrows():
        image_id = row["SEMImageID"]
        if image_id not in embeddings:
            skipped.append(image_id)
            continue

        description = get_sem_description(image_id, descriptions)

        content = render_content(row)

        content += (
            "\n\nVisual description:\n"
            + description
        )

        documents.append(
            Document(
                id=str(image_id),
                content=content,
                embedding=embeddings[image_id],
                meta=build_meta(row, include_thumbnail=include_thumbnail),
            )
        )

    if skipped:
        print(f"Warning: {len(skipped)} image(s) had metadata but no embedding, skipped: {skipped[:10]}...")
    return documents

In [ ]:
descriptions = load_descriptions(SEMDataset.DESCRIPTIONS)

In [ ]:
get_sem_description(10111, descriptions)

In [ ]:
docs = build_documents(embeds, SEMDataset.JSON, True)

In [ ]:
docs[916].meta

All 11 settings are numeric, so they all get the same "float" schema -- this is what makes range filters like `{"AP_WD": {">=": 3.0, "<=": 3.4}}` fast instead of an unindexed scan.

In [ ]:
PAYLOAD_FIELDS_TO_INDEX = [
    {"field_name": field, "field_schema": "float"}
    for field in NUMERIC_FIELDS
] + [
    {"field_name": field, "field_schema": "keyword"}
    for field in CATEGORICAL_FIELDS
] + [
    {"field_name": "SEMImageID", "field_schema": "integer"},
]

In [ ]:
PAYLOAD_FIELDS_TO_INDEX

In [ ]:
def build_collection(collection_name, embeddings_folder, embedding_dim, json_folder):
    if embedding_dim is None:
        raise ValueError(
            f"embedding_dim not set for collection '{collection_name}' -- "
            f"fill in the actual output dimension before running."
        )

    document_store = QdrantDocumentStore(
        index=collection_name,
        embedding_dim=embedding_dim,
        recreate_index=True,
        payload_fields_to_index=PAYLOAD_FIELDS_TO_INDEX,
        path=QDRANT_PATH, # local
        # url=QDRANT_URL, # server
    )

    

    embeddings = load_embeddings(embeddings_folder)
    documents = build_documents(embeddings, json_folder)
    document_store.write_documents(documents, policy=DuplicatePolicy.OVERWRITE)
    print(f"Wrote {len(documents)} documents to '{collection_name}' (dim={embedding_dim}), "
          f"indexed on {len(PAYLOAD_FIELDS_TO_INDEX)} payload fields.")
    return document_store

In [ ]:
document_store = build_collection(Qdrant.COLLECTION_QWEN, SEMDataset.EMBEDDINGS_QWEN, Qdrant.EMBEDDING_DIM_QWEN, SEMDataset.JSON)

In [ ]:
doc = document_store.get_documents_by_id(["12983"])
doc[0].content

In [ ]:
document_store.count_documents()

In [ ]:
document_store.count_documents_by_filter({"field": "meta.SEMImageID", "operator": "<=", "value": 200})

In [ ]:
def render_image_gallery(documents) -> str:
    """
    Render the retrieved candidates as inline images so the researcher can
    actually see what's being discussed, not just read a paraphrase of it.
    """
    if not documents:
        return ""

    gallery = "\n\n**Candidate images:**\n"
    for idx, doc in enumerate(documents, start=1):
        label = doc.meta.get("ImageOriginalName", doc.id)
        thumb = doc.meta.get("thumb_b64")
        gallery += f"\n{idx}. {label}\n"
        if thumb:
            gallery += f"\n![{label}]({thumb})\n"
    return gallery

In [ ]:
render_image_gallery([docs[0]])

# LAME

In [ ]:
descriptions = load_descriptions(LAMEDataset.DESCRIPTIONS)

In [ ]:
get_lame_description(LAMEDataset.CROPS / "3D-tomography__Axial__slice-001.npy", descriptions)

In [ ]:
for path in LAMEDataset.CROPS.glob("*"):
    get_lame_description(path, descriptions)

In [ ]:
NUMERIC_FIELDS = [
    "magnification_x",
    "accelerating_voltage_kv",
    "emission_current_µa",
    "dwell_time_µs",
    "working_distance_mm",
]

CATEGORICAL_FIELDS = [
    "scan_mode",
    "detector",
]

# Canonical field: (section, source key, conversion factor)
NUMERIC_FIELD_MAP = {
    "magnification_x": ("MAIN", "Magnification", 1.0),
    "accelerating_voltage_kv": ("SEM", "AcceleratorVoltage", 1e-3),
    "emission_current_µa": ("SEM", "EmissionCurrent", 1e6),
    "dwell_time_µs": ("SEM", "DwellTime", 1e6),
    "working_distance_mm": ("SEM", "WD", 1e3),
}

# Canonical field: (section, source key)
CATEGORICAL_FIELD_MAP = {
    "scan_mode": ("SEM", "ScanMode"),
    "detector": ("SEM", "Detector"),
}

FIELD_UNITS = {
    "magnification_x": "X",
    "accelerating_voltage_kv": "kV",
    "emission_current_µa": "µA",
    "dwell_time_µs": "µs",
    "working_distance_mm": "mm",
}

In [ ]:
def load_hdr_metadata(hdr_path: str | Path) -> dict:
    """
    Load and normalize the relevant metadata from a SEM .hdr file.

    Missing or invalid values are represented as None.
    """
    hdr_path = Path(hdr_path)

    parser = configparser.ConfigParser(interpolation=None)
    parser.optionxform = str  # Preserve key capitalization.

    with hdr_path.open("r", encoding="utf-8-sig") as file:
        parser.read_file(file)

    metadata = {}

    # Numeric fields and unit conversions
    for field, (section, key, factor) in NUMERIC_FIELD_MAP.items():
        raw_value = parser.get(section, key, fallback=None)

        try:
            metadata[field] = (
                float(raw_value) * factor
                if raw_value is not None and raw_value.strip()
                else None
            )
        except ValueError:
            metadata[field] = None

        metadata[f"{field}_unit"] = FIELD_UNITS[field]

    # Categorical fields
    for field, (section, key) in CATEGORICAL_FIELD_MAP.items():
        value = parser.get(section, key, fallback=None)

        metadata[field] = value.strip() if value and value.strip() else None

    return metadata

In [ ]:
hdrs = [path for path in LAMEDataset.META.glob("*")]

In [ ]:
load_hdr_metadata(hdrs[500])